In [ ]:
from datasets import load_dataset
from transformers import AutoTokenizer
from transformers import DataCollatorForTokenClassification
import numpy as np
from transformers import AutoModelForTokenClassification, TrainingArguments, Trainer
import evaluate
import json
from pathlib import Path
import random
from datasets import DatasetDict, Dataset

In [ ]:
# ! pip install datasets==4.3.0 transformers==4.57.6 spacy==3.7.5 evaluate==0.4.6 seqeval==1.2.2

In [ ]:
# python -m spice download ru_core_new_sm

In [ ]:
import spacy
from spacy.training import offsets_to_biluo_tags, biluo_to_iob
nlp = spacy.load('ru_core_new_sm')

In [ ]:
def tokenize_and_align_labels(examples):
    tokenized_inputs = tokenizer(examples["tokens"], truncation=True, is_split_into_words=True)

    labels = []
    for i, label in enumerate(examples[f"ner_tags"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)  # Map tokens to their respective word.
        previous_word_idx = None
        label_ids = []
        for word_idx in word_ids:  # Set the special tokens to -100.
            if word_idx is None:
                label_ids.append(-100)
            elif word_idx != previous_word_idx:  # Only label the first token of a given word.
                label_ids.append(label[word_idx])
            else:
                label_ids.append(-100)
            previous_word_idx = word_idx
        labels.append(label_ids)

    tokenized_inputs["labels"] = labels
    return tokenized_inputs

def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    true_predictions = [
        [label_list[p] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]
    true_labels = [
        [label_list[l] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]

    results = seqeval.compute(predictions=true_predictions, references=true_labels)
    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"],
    }

def encode_tags(example, mapping):
    example["ner_tags"] = [mapping[x] for x in example["iob_tags"]]
    return example

# data preparation

In [ ]:
# data = ...

In [ ]:
raw_ents = [] # list of like this ("Who is Shaka Khan?", {"entities": [(7, 17, "PERSON")]})

for d in data:
    ...
    raw_ents.append( (text, {"entities": spans}) )

In [ ]:
docs = []
err_count = 0
for text, annot in raw_ents:
    doc = nlp(text)
    try:
        tags = biluo_to_iob(offsets_to_biluo_tags(doc, annot['entities']))
    except:
        err_count += 1
        continue
    tags = ["O" if x == "-" else x for x in tags]
    if all(x == "O" for x in tags):
        continue
    docs.append({"text": text, "tokens": [x.text for x in doc], "iob_tags": tags})

print(err_count)

In [ ]:
random.shuffle(docs)

train_part = int(len(docs) * 0.8)
train = docs[:train_part]
test = docs[train_part:]

In [ ]:
dataset = DatasetDict({
    "train": Dataset.from_list(train),
    "test": Dataset.from_list(test),
})

In [ ]:
label_list = list({x for y in dataset["train"]["iob_tags"] for x in y})
id2label = {x:y for x,y in enumerate(label_list)}
label2id = {y:x for x,y in id2label.items()}

In [ ]:
dataset = dataset.map(lambda x: encode_tags(x, label2id), batch_size=True)

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("DeepPavlov/rubert-base-cased")


In [ ]:
data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

In [ ]:
tokenized_dataset = dataset.map(tokenize_and_align_labels, batched=True)

In [ ]:
seqeval = evaluate.load("seqeval")


In [ ]:
model = AutoModelForTokenClassification.from_pretrained("DeepPavlov/rubert-base-cased", num_labels=len(id2label), id2label=id2label, label2id=label2id)

In [ ]:
id2label

In [ ]:
training_args = TrainingArguments(
    output_dir="clause_detection_model",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=32,
    num_train_epochs=1,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    push_to_hub=False,
    report_to='none'
)

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["test"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer.train()